In [1]:
import os
import textwrap
from pathlib import Path
import fitz
from IPython.display import Markdown
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Qdrant
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from llama_parse import LlamaParse

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llama_parse_key = os.getenv("LLAMA_PARSE")


In [2]:
def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

In [4]:
pdf_text = read_pdf("data/rpp.pdf")

print(pdf_text[:1000])
document_path = Path("data/parsed_document.md")
with document_path.open("w", encoding="utf-8") as f:  # Specify utf-8 encoding
    f.write(pdf_text)

1) Onion Cultivation Guide for Tamil Nadu (Beginner Friendly)  
 
1.  Basic Information:  
      Crop Name:  Onion (Allium cepa) 
      Type:  Biennial (grown as annual for bulb production) 
      Growing Season:  August to October (Kharif), December to March (Rabi) 
      Optimal Altitude:  Grows well in plains and hilly areas up to 1200 meters above sea level. 
      Growth Duration:  Typically 4 5 months depending on the variety. 
 
2.  Soil & Land Requirements:  
      Suitable Soil Type:  Well drained sandy loam to loamy soil with rich organic matter content. 
      Soil pH Range:  6.0 to 7.5 (slightly acidic to neutral). 
      Soil Preparation:  
       Plough the field thoroughly to break up clods and achieve fine tilth. 
       Add well decomposed organic manure or compost during the final ploughing. 
       Ensure proper drainage as onions are sensitive to waterlogging. 
 
3.  Climate and Weather Requirements:  
      Temperature Range:  Optimal temperature for growth is betw

In [5]:
import requests

api_key = os.getenv("WEATHER_API")
location = "Coimbatore"
url1 = f"http://api.weatherapi.com/v1/future.json?key={api_key}&q={location}&dt=2024-12-01"

response1 = requests.get(url1)
future_weather = response1.json()


In [6]:
import os
import requests

# Get API key from environment variable
api_key = os.getenv("WEATHER_API")
location = "Coimbatore"
date = "2024-12-01"

# Construct the API request URL
url1 = f"http://api.weatherapi.com/v1/future.json?key={api_key}&q={location}&dt={date}"

# Make the request
response1 = requests.get(url1)

# Parse the response JSON
if response1.status_code == 200:
    future_weather = response1.json()
    
    # Extract total precipitation in mm from the response
    total_precip_mm = future_weather['forecast']['forecastday'][0]['day']['totalprecip_mm']
    
    # Print the result
    print(f"Total precipitation on {date} in {location}: {total_precip_mm} mm")
else:
    print(f"Failed to retrieve weather data: {response1.status_code}")


Failed to retrieve weather data: 400


In [7]:
url2 = f"http://api.weatherapi.com/v1/current.json?key={api_key}&q={location}&dt=2024-12-01"

response2 = requests.get(url2)
weather = response2.json()


In [8]:
loader = UnstructuredMarkdownLoader(document_path)
loaded_documents = loader.load()

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=128)
docs = text_splitter.split_documents(loaded_documents)

In [10]:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")
qdrant = Qdrant.from_documents(
    docs,
    embeddings,
    path="./db",
    collection_name="document_embeddings",
)

c:\Users\vishn\Desktop\Programs\CodeOClock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<?, ?it/s]


In [11]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Answer the question in the following JSON format:
{
    "answer": "<Your succinct answer here>",
    "additional_info": "<Any additional helpful information, if applicable>",
    "sources": "<Provide any source names or details, if applicable>"
}

Ensure the JSON format is valid and complete.
"""


parser = LlamaParse(
    api_key=llama_parse_key,
    result_type="markdown",
    parsing_instruction="The provided document contains detailed tax information.",
    max_timeout=5000,
)

def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))


In [12]:
llm = ChatGroq(temperature=0, model_name="llama-3.1-70b-versatile")
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=qdrant.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": True, "output_key": "answer"},
)


In [15]:
recommendation_prompt = """Based on the provided information about the location and its agricultural conditions, recommend suitable crops that can be cultivated in that area.

Location: Coimbatore, Tamil Nadu, India
Soil Type: Red loamy soil
Climate: Tropical wet and dry climate
Season: Kharif and Rabi
Local Agricultural Practices: Onion, Banana, and Coconut cultivation

Recommendations:

- Consider the following factors when recommending crops:
    - Soil nutrient levels and pH
    - Average rainfall and irrigation availability
    - Temperature range during the growing season
    - Market demand for certain crops in the area

List the recommended crops with a brief explanation of why each crop is suitable for the given conditions.
"""
parser = LlamaParse(
    api_key=llama_parse_key,
    result_type="markdown",
    parsing_instruction="The provided document contains detailed crop location, including unaudited crop information.",
    max_timeout=5000,
)

def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))

llm = ChatGroq(temperature=0, model_name="llama-3.1-70b-versatile")
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=qdrant.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": True},
)

response = qa.invoke("What are the recommended crops?")
print_response(response)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: Soil & Land Requirements: 
    Suitable Soil Type: Well-drained sandy loam or loamy soil with good organic matter. 
    Soil pH Range: 6.0 to 6.8 (slightly acidic to neutral). 
    Soil Preparation: 
        Plough the field thoroughly and incorporate organic matter (like compost). 
        Level the field to ensure good drainage.

Climate and Weather Requirements: 
    Temperature Range: 20°C to 30°C is ideal for optimal growth. 
    Rainfall Requirement: Moderate rainfall; ensure good irrigation during dry periods (about 600-

800 mm during the growing season). Sunlight: Requires full sunlight (6-8 hours of direct sunlight per day). Wind Sensitivity: Sensitive to strong winds; consider windbreaks if necessary

In [14]:
prompt_template = """
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Answer the question and provide additional helpful information,
based on the pieces of information, if applicable. Be succinct.

Responses should be properly formatted in JSON, dont add sentances. Use concise language.
"""

parser = LlamaParse(
    api_key=llama_parse_key,
    result_type="markdown",
    parsing_instruction="The provided document contains detailed crop information, including unaudited crop information.",
    max_timeout=5000,
)

def print_response(response):
    response_txt = response["result"]
    for chunk in response_txt.split("\n"):
        if not chunk:
            print()
            continue
        print("\n".join(textwrap.wrap(chunk, 100, break_long_words=False)))

llm = ChatGroq(temperature=0, model_name="llama-3.1-70b-versatile")
prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=qdrant.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt, "verbose": True},
)

response = qa.invoke("What is the fertilizer for brinjal?")
print_response(response)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: 4. Water Requirements & Irrigation: 
Water Needs:   Brinjal needs regular watering, especially during dry spells. 
Recommended Irrigation Methods:   
  Drip Irrigation:   Helps in water conservation and maintains consistent soil moisture. 
  Furrow Irrigation:   Commonly used in Tamil Nadu; irrigate the furrows between plant rows. 
Irrigation Schedule:   
  Seedling Stage:   Keep the soil consistently moist. 
  Vegetative Stage:   Water regularly to maintain even soil moisture. 
  Flowering and Fruiting Stage:   Crucial to maintain consistent watering to promote fruit

development. Final Maturity: Gradually reduce watering as fruits ripen, but do not let the soil dry completely.

 5. Fertilizer & Nutrient Needs